# Fold map: what shows on a folded origami model, and where it comes from

When you fold a model, most of the paper ends up hidden inside. This notebook works out exactly which parts of the square stay visible, and where each visible point lands on the folded model. That **fold map** is the foundation for printing images on paper that only come together once it's folded, and eventually for designing illusions: a sheet that reads as one picture flat and another folded, or images hidden where nobody looks.

It works for any model that folds **flat** and that [Flat-Folder](https://github.com/origamimagiro/flat-folder) can solve. The traditional crane is loaded by default.

1. **Where each piece lands:** reflecting faces across creases.
2. **What's on top:** the stacking order of layers.
3. **The visibility map:** which parts of each side of the sheet show from the front and back.
4. **Which way is up:** how each visible region is rotated on the folded model.
5. **Decal:** print a front picture and a back picture so they appear on the folded model.
6. **Building blocks** for going further.

Nothing here needs a GPU. On Colab, the default CPU runtime is fine.

## Setup

The code lives in [rath-hy/folded-illusions](https://github.com/rath-hy/folded-illusions/tree/main/fold-map). On Colab this clones it; if you're running the notebook from inside that folder, it uses the local files.

In [ ]:
import importlib, os, subprocess, sys

FOLDER = "fold-map"
if not os.path.exists("fold_map.py"):
    if not os.path.exists("folded-illusions"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/rath-hy/folded-illusions.git"], check=True)
    os.chdir(os.path.join("folded-illusions", FOLDER))
if "google.colab" in sys.modules:
    # On Colab, rerunning this cell gets the latest code and reloads anything already imported.
    subprocess.run(["git", "pull", "-q"], check=True)
    for name in ["fold_map", "directions", "decal", "illusion", "story"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from fold_map import FoldedModel, plot_folded_views, plot_visibility, summarize
from directions import plot_directions
from decal import fit, make_sheets, picture_aspect, plot_preview, render_side, simulate, test_picture

## Load a model

Two files describe a flat-folded model, both in the [FOLD format](https://github.com/edemaine/fold):

- **The crease pattern:** the fold lines on the unfolded square, marked mountain or valley.
- **The folded state:** where every vertex ends up after folding, plus which of any two overlapping faces is on top.

To use your own model, open [Flat-Folder](https://origamimagiro.github.io/flat-folder/), load a crease pattern (or pick an example), press **Fold**, and if it finds several states, pick the one that matches the real model with the Component and State controls. Then **Export → cp** and **Export → state**, and upload both below. The code checks that the two files describe the same pattern.

In [ ]:
CREASE_PATTERN = "004_traditional_Crane_cp.fold"
FOLDED_STATE = "004_traditional_Crane_state.fold"
UPLOAD_MODEL = False     # True: upload your own crease pattern and folded state (Colab)

if UPLOAD_MODEL:
    from google.colab import files
    uploaded = list(files.upload())
    FOLDED_STATE = next(name for name in uploaded if "state" in name.lower())
    CREASE_PATTERN = next(name for name in uploaded if name != FOLDED_STATE)

model = FoldedModel(CREASE_PATTERN, FOLDED_STATE)
print(f"{len(model.faces)} faces, {len(model.edges)} creases and edges, "
      f"{int((model.normal < 0).sum())} faces end up flipped over")

## 1. Where each piece lands

The creases cut the sheet into flat faces, and folding moves each face rigidly: a rotation, a shift, and a reflection if it ends up flipped over. Comparing a face's corners on the sheet and in the folded state gives that transform exactly (`model.to_folded[f]`, and its inverse `model.to_sheet[f]`).

Below, a rainbow test texture is "printed" on the sheet, and the folded model is drawn from both sides using those transforms, showing the topmost layer at every point. Each colored patch tells you which part of the sheet you're looking at. Side 2 is drawn paler than side 1.

**Conventions** follow Flat-Folder's display: *side 1* is the side of the sheet you see in its crease pattern view, *front* is its folded view, and *back* is the folded view with **Flip** checked (the model turned over left to right).

In [ ]:
side1, side2, sheet_bounds, unknown_sheet = model.sheet_visibility(800)
views, folded_bounds, unknown_views = model.views(900)
plot_folded_views(model, views, folded_bounds, sheet_bounds);

## 2. What's on top

Where several layers pile up, the one closest to you is what you see. Working out a valid stacking order for a whole crease pattern is NP-hard in general, and a pattern can have several valid orders (the traditional crane has 5). Flat-Folder solves it and stores the answer as `faceOrders`: triples `[f, g, s]` saying face `f` is on the side of face `g` that **g's** normal points to (`s = +1`) or the opposite side (`s = -1`). It's easy to read that the wrong way around; the first version of this code did, and the result was an impossible cycle where no face was on top.

`FoldedModel` turns those pairs into one "is above" relation and refuses contradictory input. Here's the stack at the thickest point of the model, listed by face number from front to back:

In [ ]:
def stack_at(model, point):
    # Faces covering a folded point, ordered from the front view (bottommost) to the back view (topmost).
    faces = np.flatnonzero(model.cover(np.array([point]))[0])
    return sorted(faces, key=lambda f: sum(model.higher[f, g] for g in faces if g != f))

lo, hi = folded_bounds
Q = np.stack(np.meshgrid(np.linspace(lo[0], hi[0], 120), np.linspace(lo[1], hi[1], 120)), -1).reshape(-1, 2)
thickest = Q[model.cover(Q).sum(1).argmax()]
stack = stack_at(model, thickest)
print(f"{len(stack)} layers at folded point ({thickest[0]:.3f}, {thickest[1]:.3f}), front to back:")
print("  faces", [int(f) for f in stack])

## 3. The visibility map

For every point of the sheet: find where it lands, then check whether its face is the frontmost layer there (visible from the front) or the backmost (visible from the back). Each side of the sheet is drawn as seen from that side.

In [ ]:
summarize(model, side1, side2, unknown_sheet + unknown_views)
plot_visibility(model, side1, side2, sheet_bounds);

For the crane, side 1 is entirely hidden: fold it from origami paper colored on one side and no white shows. About a quarter of side 2 is visible, split evenly between front and back, and some visible boundaries cut across faces, wherever the edge of a flap covers part of the face beneath it. Folding a real crane, shading what shows, and unfolding it reproduces this map.

## 4. Which way is up

To show a picture upright on the folded model, each visible region of the sheet has to be printed rotated (and, in principle, mirrored) by its face's transform. Each **F** below is drawn the way it must be printed on that region to read upright when folded. A backwards F would mean that region shows a mirror image.

In [ ]:
fig, mirrored = plot_directions(model, side1, side2)
print("regions needing a mirrored picture:", "some" if mirrored else "none")

For the crane, every region only needs rotating, by multiples of 22.5°, the crane's basic angle. Neighboring regions often need very different rotations, so a picture can't be placed region by region by hand. The decal below maps it point by point instead.

## 5. Decal: a picture on each side of the folded model

Each picture is center-cropped to the folded silhouette's bounding box as seen from that side, and every visible point of the sheet takes the picture's color at the spot where it lands. Only visible areas are printed, plus a thin **bleed** past their edges so slightly imperfect folds don't reveal white slivers. If the model shows both sides of the paper, you get one sheet per side to print double-sided.

Tiny gray letters **A–D** sit in hidden areas near the corners. The preview marks where each corner ends up on the folded model, so you know which corner becomes what (for the crane: the tail, a wing, the head, and the other wing). Fold with the printed side out and the corners in those roles.

Leave the pictures as `None` for test pictures (a big F and B), or upload your own. Rendering the full-size sheet takes about a minute.

In [ ]:
FRONT_PICTURE = None     # path to an image, or None for the test picture
BACK_PICTURE = None
UPLOAD_PICTURES = False  # True: upload the front picture, then the back picture (Colab)
SHEET_PIXELS = 2400      # 2400 = an 8-inch square at 300 dpi
BLEED = 0.01             # margin printed past visible areas, as a fraction of the sheet

if UPLOAD_PICTURES:
    from google.colab import files
    print("front picture:"); FRONT_PICTURE = list(files.upload())[0]
    print("back picture:"); BACK_PICTURE = list(files.upload())[0]

aspect = picture_aspect(model)
print(f"pictures are cropped to {aspect:.2f}:1 (width:height)")
pictures = {"front": fit(Image.open(FRONT_PICTURE) if FRONT_PICTURE else test_picture("F", (196, 120, 60)), aspect),
            "back": fit(Image.open(BACK_PICTURE) if BACK_PICTURE else test_picture("B", (52, 98, 168)), aspect)}

sheets = make_sheets(model, pictures, SHEET_PIXELS, BLEED)
simulated, bounds = simulate(model, sheets)
plot_preview(model, pictures, simulated, bounds);

The sheets to print, each as seen from its own side. Print at **actual size** (not "fit to page") and cut along the thin gray border.

In [ ]:
fig, axes = plt.subplots(1, len(sheets), figsize=(7 * len(sheets), 7), squeeze=False)
for ax, (side, sheet) in zip(axes[0], sheets.items()):
    ax.imshow(sheet["image"]); ax.axis("off")
    ax.set_title(f"side {side}: {sheet['printed'].mean():.0%} of the sheet printed")
plt.show()

for side, sheet in sheets.items():
    name = f"decal_side{side}.png"
    sheet["image"].save(name, dpi=(SHEET_PIXELS / 8, SHEET_PIXELS / 8))
    print("saved", name)
    try:
        from google.colab import files
        files.download(name)
    except ImportError:
        pass

## 6. Building blocks

Everything above is assembled from a few pieces you can reuse:

| | |
|---|---|
| `model.to_folded[f]`, `model.to_sheet[f]` | 3×2 affine maps for face `f`; apply with `fold_map.apply(M, points)` |
| `model.normal[f]` | `+1` if face `f` keeps side 2 facing +z after folding, `-1` if it flipped |
| `model.higher[f, g]` | face `f` is above face `g` (toward +z) wherever they overlap |
| `model.cover(points)`, `model.stack_ends(cover)` | which faces contain folded points, and the frontmost and backmost of them |
| `side1`, `side2` | the visibility map of each side: 0 hidden, 1 front, 2 back |
| `views[view]` | for every pixel of the folded front or back view: the sheet point shown, its side, and its face |
| `render_side(model, pictures, side, res, bleed)` | one side's colors, visible mask and printed mask |

Some directions this opens up:

- **Hidden images:** the hidden areas are free canvas. Fill them with a third picture that only appears when the model is unfolded.
- **Flat vs. folded illusions:** make the whole flat sheet read as one picture while the folded model shows another, by optimizing the colors of visible regions to satisfy both.
- **Non-flat models:** a crane with spread wings is 3D. A 3D folded mesh (for example from Origami Simulator) and a renderer would replace the front/back views.
- **Measured fold maps:** print a coded pattern, fold, photograph, and decode which sheet points are visible, then compare with the computed map.

## References

- H. A. Akitaya, E. D. Demaine, J. S. Ku. [Computing Flat-Folded States](https://erikdemaine.org/papers/FlatFolder_OSME2024/). *Origami⁸ (OSME)*, 2024. The algorithm behind Flat-Folder.
- J. S. Ku. [Flat-Folder](https://github.com/origamimagiro/flat-folder), software.
- E. D. Demaine, J. S. Ku, R. J. Lang. [FOLD file format specification](https://github.com/edemaine/fold/blob/main/doc/spec.md).
- M. Bern, B. Hayes. The complexity of flat origami. *SODA*, 1996. Flat foldability is NP-hard.